# Reproducibility lane A — deterministic private 500

Runs the same deterministic 500-row private subset as lane B. Input questions remain local; only the output directory differs. Start from a fresh A100 runtime and an unused Drive output directory.

In [ ]:
# Cell 1 — Clone the exact deterministic code revision and install. Restart once afterward.
import subprocess,sys
from pathlib import Path
CODE_COMMIT="2254a166b1f607a9fb7a7d55b1136c0238aa21a1"
REPO=Path("/content/qwen-math-final-2026")
if not REPO.exists():
    subprocess.run(["git","clone","https://github.com/jhparktime/qwen-math-final-2026.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"fetch","origin",CODE_COMMIT],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--detach",CODE_COMMIT],check=True)
actual=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip();assert actual==CODE_COMMIT,(actual,CODE_COMMIT)
subprocess.run([sys.executable,"-m","pip","install","-q","--no-cache-dir","-r",str(REPO/"requirements-colab.txt")],check=True)
subprocess.run([sys.executable,"-m","pip","uninstall","-q","-y","torchcodec"],check=False)
print("[CODE COMMIT]",actual);print("[SETUP] restart runtime once, then continue at Cell 2")

In [ ]:
# Cell 2 — Restore paths and mount Drive.
from pathlib import Path
REPO=Path("/content/qwen-math-final-2026");CODE_COMMIT="2254a166b1f607a9fb7a7d55b1136c0238aa21a1"
assert REPO.exists(),REPO
import subprocess
actual=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip();assert actual==CODE_COMMIT,(actual,CODE_COMMIT)
from google.colab import drive
drive.mount("/content/drive")
print("[CODE COMMIT]",actual)

In [ ]:
# Cell 3 — Cache the pinned base model. No test data is read or sent.
import os
from huggingface_hub import snapshot_download
os.environ.pop("HF_HUB_OFFLINE",None);os.environ.pop("TRANSFORMERS_OFFLINE",None)
BASE_MODEL="Qwen/Qwen2.5-3B-Instruct";MODEL_REVISION="aa8e72537993ba99e69dfaafa59ed015b17504d1"
model_cache=snapshot_download(repo_id=BASE_MODEL,revision=MODEL_REVISION)
print("[BASE MODEL CACHED]",model_cache)

In [ ]:
# Cell 4 — Build the identical deterministic 500-row subset and lane-A config.
import hashlib,json
import pandas as pd
SOURCE_INPUT=Path("/content/drive/MyDrive/test_submission.csv")
ADAPTER_PATH=Path("/content/drive/MyDrive/2026소중한챌린지/runs/RFT-0008D-r3mix-r2continue-r16-a100/adapter_final")
OUTPUT_DIR=Path("/content/drive/MyDrive/2026소중한챌린지/runs/REPRO-0001A-private500-deterministic")
INPUT_PATH=Path("/content/private_repro500.csv");CONFIG_PATH=Path("/content/repro500_A_config.json")
assert SOURCE_INPUT.exists(),SOURCE_INPUT;assert (ADAPTER_PATH/"adapter_config.json").exists(),ADAPTER_PATH
frame=pd.read_csv(SOURCE_INPUT,dtype=str,keep_default_na=False);frame.columns=[str(c).lstrip("\ufeff").strip() for c in frame.columns]
assert len(frame)==2000 and {"id","question"}.issubset(frame.columns) and frame.id.is_unique
assert "answer" not in frame.columns or (frame.answer.str.strip()=="").all()
frame["_row"]=range(len(frame));frame["_rank"]=frame.id.map(lambda qid:hashlib.sha256(f"3|deterministic_repro500|{qid}".encode()).hexdigest())
sample=frame.sort_values("_rank").head(500).sort_values("_row")[["id","question"]].reset_index(drop=True)
sample.to_csv(INPUT_PATH,index=False,encoding="utf-8")
cfg=json.loads((REPO/"configs/final_inference.json").read_text(encoding="utf-8"));cfg["run_id"]="REPRO-0001A-private500-deterministic";cfg["expected_rows"]=500
CONFIG_PATH.write_text(json.dumps(cfg,ensure_ascii=False,indent=2),encoding="utf-8")
input_sha=hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest();print("[SUBSET]",len(sample),input_sha);print("[OUTPUT]",OUTPUT_DIR)

In [ ]:
# Cell 5 — Offline deterministic inference with live logs.
import os,subprocess,sys
env=os.environ.copy();env["PYTHONPATH"]=str(REPO);env["HF_HUB_OFFLINE"]="1";env["TRANSFORMERS_OFFLINE"]="1"
command=[sys.executable,str(REPO/"inference/final_inference.py"),"--input",str(INPUT_PATH),"--adapter",str(ADAPTER_PATH),"--output-dir",str(OUTPUT_DIR),"--config",str(CONFIG_PATH)]
process=subprocess.Popen(command,cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
for line in process.stdout:print(line,end="",flush=True)
exit_code=process.wait()
if exit_code:raise RuntimeError(f"Inference failed with exit code {exit_code}")

In [ ]:
# Cell 6 — Validate lane A and compare automatically when lane B is ready.
import hashlib
SUBMISSION=OUTPUT_DIR/"submissions/submission.csv"
subprocess.run([sys.executable,str(REPO/"scripts/validate_submission.py"),"--input",str(INPUT_PATH),"--submission",str(SUBMISSION),"--expected-rows","500"],cwd=REPO,check=True)
OTHER=Path("/content/drive/MyDrive/2026소중한챌린지/runs/REPRO-0001B-private500-deterministic/submissions/submission.csv")
def sha(path):return hashlib.sha256(path.read_bytes()).hexdigest()
print("[A SHA]",sha(SUBMISSION))
if OTHER.exists():
    a=pd.read_csv(SUBMISSION,dtype=str,keep_default_na=False);b=pd.read_csv(OTHER,dtype=str,keep_default_na=False);assert a.id.tolist()==b.id.tolist()
    changed=int((a.answer!=b.answer).sum());print("[B SHA]",sha(OTHER));print("[EXACT FILE MATCH]",SUBMISSION.read_bytes()==OTHER.read_bytes());print("[DIFFERENT ANSWERS]",changed);assert changed==0,changed
else:print("[WAIT] lane B has not finished yet; rerun this cell after B completes.")